In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install transformers

In [57]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import shutil
import string
from sklearn.metrics import classification_report
import tensorflow as tf
from keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras import losses
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping,ModelCheckpoint
from tensorflow.keras.layers import Dense, Input, Dropout, Bidirectional, LSTM, Embedding, BatchNormalization,  Reshape, Conv2D, MaxPool2D, concatenate, Flatten, Activation
import torch
import numpy as np
from transformers import BertTokenizer, BertModel,RobertaTokenizer, RobertaModel,AutoTokenizer, AutoModel

import ast

In [4]:
# Verificar si la GPU está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
archivo_3 = '/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/Copia de ExternData.csv'
train_full = pd.read_csv(archivo_3)

In [6]:
train_full

,Problem Description,Python Code,Source,Description Error,Error Label
0,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2
1,Sum numbers from 1 to n,def sum_to_n_buggy(n):\n total = 0\n i =...,Based on typical mistakes in Codeforces common...,i is incorrectly incremented,2
2,Prompt user for numbers until 0 is entered,def input_until_zero_buggy():\n num = int(i...,Common error from GeeksforGeeks Python while l...,number is never updated in the loop,2
3,Find first multiple of 7 greater than n,def first_multiple_7_buggy(n):\n i = n + 1\...,Typical error in CodeNet beginner loop problems,i not incremented,2
4,Calculate factorial of n,def factorial_buggy(n):\n result = 1\n w...,Inspired by classic factorial exercises,"decrement by 2, not 1",2
5,Count digits in a positive integer n,def count_digits_buggy(n):\n count = 0\n ...,Inspired by digit counting problems in Codefor...,n not updated inside the loop,2
6,Sum all even numbers up to n,def sum_evens_buggy(n):\n total = 0\n i ...,"CodeChef/Codeforces summing problems, conditio...",loop condition incorrect,1
7,Find a prime number using break,def find_prime_buggy():\n num = 2\n whil...,Inspired by break/continue loop errors from Py...,break outside the if block,2
8,Print cubes of numbers from 1 to n,def print_cubes_buggy(n):\n while i <= n:\n...,"Common initialization error, based on CodeNet ...",i not initialized,0
9,Prompt for password until correct,def prompt_password_buggy():\n password = i...,Typical validation loop error from Python begi...,password requested only once,2


# AST spliter D. Gries form

In [7]:
class WhileLoopFinder(ast.NodeVisitor):
    def __init__(self, source_code):
        self.source_code = source_code.splitlines()
        self.functions_with_while = []
        self.current_function = None

    def visit_FunctionDef(self, node):
        original_current_function = self.current_function
        self.current_function = {
            "function_name": node.name,
            "has_while_loop": False,
            "pre_while_code_lines": [],
            "while_loops": []
        }

        function_start_line = node.lineno - 1
        function_end_line = (node.end_lineno if hasattr(node, 'end_lineno') else len(self.source_code))
        function_lines = self.source_code[function_start_line:function_end_line]

        found_while = False
        for stmt in node.body:
            if isinstance(stmt, ast.While):
                self.current_function["has_while_loop"] = True
                found_while = True
                try:
                    while_condition = ast.unparse(stmt.test).strip()
                    print("OK condition")
                except AttributeError:
                    print("Error al obtener la condición del while")
                try:
                    while_body_code = ast.unparse(ast.Module(body=stmt.body, type_ignores=[])).strip()
                    print("Ok body")
                except AttributeError:
                    print("Error al obtener el código del cuerpo del while")

                self.current_function["while_loops"].append({
                    "condition": while_condition,
                    "body_code": while_body_code
                })
            elif not found_while:
                start_line = stmt.lineno - 1
                end_line = (stmt.end_lineno if hasattr(stmt, 'end_lineno') else stmt.lineno)
                self.current_function["pre_while_code_lines"].extend(self.source_code[start_line:end_line])
            self.generic_visit(stmt)

        if self.current_function["has_while_loop"]:
            last_while_end_line = None
            if self.current_function["while_loops"]:
                last_while_node = next((node for node in reversed(node.body) if isinstance(node, ast.While)), None)
                if last_while_node and hasattr(last_while_node, 'end_lineno'):
                    last_while_end_line = last_while_node.end_lineno
            post_while_code_lines = []
            if last_while_end_line:
                function_indent = len(self.source_code[node.lineno - 1]) - len(self.source_code[node.lineno - 1].lstrip())
                for line in self.source_code[last_while_end_line:function_end_line]:
                    if line.startswith(self.source_code[node.lineno - 1][:function_indent] + "    "):
                        post_while_code_lines.append(line[function_indent + 4:])
                    else:
                        post_while_code_lines.append(line[function_indent:])
            self.current_function["post_while_code_lines"] = "\n".join(post_while_code_lines).strip()
            self.current_function["pre_while_code_lines"] = "\n".join(self.current_function["pre_while_code_lines"]).strip()
            self.functions_with_while.append(self.current_function)

        self.current_function = original_current_function



In [8]:
def DGries_states(solution):
  try:
    # Crear el AST
    tree = ast.parse(solution)
    # Recorrer el AST con nuestro visitante
    finder = WhileLoopFinder(solution)
    finder.visit(tree)
    for func_info in finder.functions_with_while:
      initial_state=func_info["pre_while_code_lines"] if func_info["pre_while_code_lines"] else False
      end_state=""
      transformation_state=""
      for j, wl in enumerate(func_info["while_loops"]):
          end_state=wl["condition"]
          transformation_state=wl["body_code"]

      if not initial_state:
        initial_state=transformation_state
      return initial_state,transformation_state,end_state
  except Exception as error:
    return "Exception:"+str(error),"Exception:"+str(error),"Exception:"+str(error)




# Encoder Description and Code

In [9]:
class EncodeTextSource:
  def __init__(self):
    self.is_loadtokenizers=False



  def tokenize_and_generate_embeddings_descriptions(self,description):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input description and move tokens to GPU
    tokens = self.beart_tokenizer.encode_plus(description, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.beart_model(**tokens)
    # Extract embeddings for all tokens
    desc_embeddings = outputs.last_hidden_state.cpu().numpy()
    return desc_embeddings

  def tokenize_and_generate_embeddings_codes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.beart_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.codebeart_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()

    return code_embeddings

  def load_beart_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    model_name = "bert-base-uncased"
    self.beart_model = BertModel.from_pretrained(model_name)
    self.beart_tokenizer = BertTokenizer.from_pretrained(model_name)
    self.beart_model.to(device)

  def load_codebert_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    self.codebeart_tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
    self.codebeart_model = RobertaModel.from_pretrained("microsoft/codebert-base")
    self.codebeart_model.to(device)

  def start_tokenizer(self):
    self.load_beart_tokenizer()
    self.load_codebert_tokenizer()
    self.is_loadtokenizers=True




# All characteristics

In [10]:
train_full.head()

,Problem Description,Python Code,Source,Description Error,Error Label
0,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2
1,Sum numbers from 1 to n,def sum_to_n_buggy(n):\n total = 0\n i =...,Based on typical mistakes in Codeforces common...,i is incorrectly incremented,2
2,Prompt user for numbers until 0 is entered,def input_until_zero_buggy():\n num = int(i...,Common error from GeeksforGeeks Python while l...,number is never updated in the loop,2
3,Find first multiple of 7 greater than n,def first_multiple_7_buggy(n):\n i = n + 1\...,Typical error in CodeNet beginner loop problems,i not incremented,2
4,Calculate factorial of n,def factorial_buggy(n):\n result = 1\n w...,Inspired by classic factorial exercises,"decrement by 2, not 1",2


In [15]:
train_full['Python Code']=train_full['Python Code'].apply(lambda x: x.replace("\\n","\n"))

In [16]:
train_full[["Estado incial","Transformación de estado","Estado final"]]=train_full['Python Code'].apply(DGries_states).apply(pd.Series)

OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body


In [17]:
train_full

,Problem Description,Python Code,Source,Description Error,Error Label,Estado incial,Transformación de estado,Estado final
0,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2,i = 0\n count = 0,if i % 2 == 0:\n count += 1,i <= n
1,Sum numbers from 1 to n,def sum_to_n_buggy(n):\n total = 0\n i =...,Based on typical mistakes in Codeforces common...,i is incorrectly incremented,2,total = 0\n i = 1,total += i\ni = i + 0,i <= n
2,Prompt user for numbers until 0 is entered,def input_until_zero_buggy():\n num = int(i...,Common error from GeeksforGeeks Python while l...,number is never updated in the loop,2,num = int(input('Enter number (0 to exit): ')),"print('Number:', num)",num != 0
3,Find first multiple of 7 greater than n,def first_multiple_7_buggy(n):\n i = n + 1\...,Typical error in CodeNet beginner loop problems,i not incremented,2,i = n + 1,pass,i % 7 != 0
4,Calculate factorial of n,def factorial_buggy(n):\n result = 1\n w...,Inspired by classic factorial exercises,"decrement by 2, not 1",2,result = 1,result *= n\nn -= 2,n > 1
5,Count digits in a positive integer n,def count_digits_buggy(n):\n count = 0\n ...,Inspired by digit counting problems in Codefor...,n not updated inside the loop,2,count = 0,n % 10\ncount += 1,n > 0
6,Sum all even numbers up to n,def sum_evens_buggy(n):\n total = 0\n i ...,"CodeChef/Codeforces summing problems, conditio...",loop condition incorrect,1,total = 0\n i = 2,total += i\ni += 2,i < n
7,Find a prime number using break,def find_prime_buggy():\n num = 2\n whil...,Inspired by break/continue loop errors from Py...,break outside the if block,2,num = 2,is_prime = True\ndivisor = 2\nwhile divisor < ...,True
8,Print cubes of numbers from 1 to n,def print_cubes_buggy(n):\n while i <= n:\n...,"Common initialization error, based on CodeNet ...",i not initialized,0,print(i ** 3)\ni += 1,print(i ** 3)\ni += 1,i <= n
9,Prompt for password until correct,def prompt_password_buggy():\n password = i...,Typical validation loop error from Python begi...,password requested only once,2,password = input('Enter password: '),print('Incorrect'),password != 'secret'


In [18]:
encoder=EncodeTextSource()
encoder.start_tokenizer()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [21]:
%%time
problem=train_full['Problem Description'].apply(encoder.tokenize_and_generate_embeddings_descriptions).to_numpy()
code=train_full['Python Code'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()
startstate = train_full['Estado incial'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()
finalstate = train_full['Estado final'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()
transstate = train_full['Transformación de estado'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()

CPU times: user 1.91 s, sys: 43.6 ms, total: 1.95 s
Wall time: 2.66 s


In [22]:
problem.shape,code.shape,startstate.shape,finalstate.shape,transstate.shape

((35,), (35,), (35,), (35,), (35,))

In [24]:
print(startstate.shape)
print(startstate[0].shape)


(35,)
(1, 8, 768)


In [25]:
Xp=np.array([sentence[0].mean(axis=0) for sentence in problem])
Xcode=np.array([sentence[0].mean(axis=0) for sentence in code])
Xs=np.array([sentence[0].mean(axis=0) for sentence in startstate])
Xf=np.array([sentence[0].mean(axis=0) for sentence in finalstate])
Xt=np.array([sentence[0].mean(axis=0) for sentence in transstate])
Xp.shape,Xs.shape,Xf.shape,Xt.shape

((35, 768), (35, 768), (35, 768), (35, 768))

In [27]:
y=train_full['Error Label']
y=y.to_numpy()
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

# Keras model

In [30]:
from tensorflow.keras import backend as K
import gc
def ANNTC(input,base,pow_initial,num_max_blocks):

  drop_out=0.5
  print("base",base,"pow",pow_initial,"num_max_blocks",num_max_blocks)
  n=num_max_blocks//2
  neurons=int(base**(pow_initial+n+1))
  # Encoder
  x=None
  print("Encoder",n)
  try:
    for i in range(n):
      x=Dense(neurons)(input)
      x=BatchNormalization()(x)
      x=Activation('relu')(x)
      x=Dropout(drop_out)(x)
      input=x
      drop_out=0.2
      print("Block ",i,neurons)
      neurons=int(neurons/base)



    #BottleNeck
    x=Dense(neurons)(x)
    x=BatchNormalization()(x)
    x=Activation('relu')(x)
    x=Dropout(0.2)(x)
    print("BottleNeck",neurons)

    # Decoder
    print("Decoder",n)
    for i in range(n):
      neurons=int(neurons*base)
      print("Block ",i,neurons)
      x=Dense(neurons)(x)
      x=BatchNormalization()(x)
      x=Activation('relu')(x)
      x=Dropout(drop_out)(x)
      if i==n-1:
        drop_out=0.5
      drop_out=0.2
  except Exception as err:

    K.clear_session()
    gc.collect()
    del x
    print(f"Unexpected {err=}, {type(err)=}")
    raise

  return x

def neural_network_all(problem,code,start_input,trass_input,final_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_p=ANNTC(problem,base,pow_initial,num_max_blocks)
  mlp_code=ANNTC(code,base,pow_initial,num_max_blocks)
  mlp_s=ANNTC(start_input,base,pow_initial,num_max_blocks)
  mlp_t=ANNTC(trass_input,base,pow_initial,num_max_blocks)
  mlp_f=ANNTC(final_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_p,mlp_code ,mlp_s, mlp_t,mlp_f])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[problem_input,code,start_input, trass_input,final_input], outputs=output)
  return model

def neural_network_code_gries(code_input,start_input,trass_input,final_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_code=ANNTC(code_input,base,pow_initial,num_max_blocks)
  mlp_s=ANNTC(start_input,base,pow_initial,num_max_blocks)
  mlp_t=ANNTC(trass_input,base,pow_initial,num_max_blocks)
  mlp_f=ANNTC(final_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_code, mlp_s, mlp_t,mlp_f])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[code_input, start_input, trass_input,final_input], outputs=output)
  return model


def neural_network(problem,start_input,trass_input,final_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_p=ANNTC(problem,base,pow_initial,num_max_blocks)
  mlp_s=ANNTC(start_input,base,pow_initial,num_max_blocks)
  mlp_t=ANNTC(trass_input,base,pow_initial,num_max_blocks)
  mlp_f=ANNTC(final_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_p, mlp_s, mlp_t,mlp_f])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[problem_input, start_input, trass_input,final_input], outputs=output)
  return model

def neural_network_wproblem(start_input,trass_input,final_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_s=ANNTC(start_input,base,pow_initial,num_max_blocks)
  mlp_t=ANNTC(trass_input,base,pow_initial,num_max_blocks)
  mlp_f=ANNTC(final_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_s, mlp_t,mlp_f])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[start_input, trass_input,final_input], outputs=output)
  return model

def neural_network_problem_code(problem,code_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_p=ANNTC(problem,base,pow_initial,num_max_blocks)
  mlp_code=ANNTC(code_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_p, mlp_code])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[problem, code_input], outputs=output)
  return model

def neural_network_code(code_input,base=10,pow_initial=4,num_max_blocks=3):

  mlp_code=ANNTC(code_input,base,pow_initial,num_max_blocks)
  ## output layer
  output=Dense(8)(mlp_code)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[code_input], outputs=output)
  return model

In [32]:
models=[]
models_names=["problemall","problem_gries","code_gries","gries","problem_code","code"]

problem_input=Input((768,))
code_input=Input((768,))
start_input=Input((768,))
trass_input=Input((768,))
final_input=Input((768,))

print("problem all")
models.append(neural_network_all(problem_input,code_input,start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3))
print("problem gries")
models.append(neural_network(problem_input,start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3))
print("problem code gries")
models.append(neural_network_code_gries(code_input,start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3))
print("gries")
models.append(neural_network_wproblem(start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3))
print("problem code")
models.append(neural_network_problem_code(problem_input,code_input,base=8,pow_initial=1, num_max_blocks=3))
print("code")
models.append(neural_network_code(code_input,base=8,pow_initial=1, num_max_blocks=3))


problem all
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
problem gries
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
problem code gries
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
Bo

In [33]:
from sklearn.metrics import matthews_corrcoef, precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

def calculate_mcc_multiclass(y_true, y_pred_probs):
    y_pred_labels = np.argmax(y_pred_probs, axis=1)
    # Convertir etiquetas verdaderas one-hot a etiquetas enteras si es necesario
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    return matthews_corrcoef(y_true, y_pred_labels)

def calculate_auc_pr_multiclass(y_true, y_pred_probs, average='macro'):
    n_classes = y_pred_probs.shape[1]
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    y_true_bin = label_binarize(y_true, classes=range(n_classes))

    auc_pr_list = []
    for i in range(n_classes):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_probs[:, i])
        auc_pr_list.append(auc(recall, precision))

    if average == 'macro':
        return np.mean(auc_pr_list)
    elif average == 'weighted':
        class_counts = y_true_bin.sum(axis=0)
        return np.average(auc_pr_list, weights=class_counts)
    else:
        return auc_pr_list


In [37]:
from tensorflow import keras
from time import time

accuracies_predict=[]
loss_predict=[]
times_predict=[]
mccs_predict=[]
aucpr_predict=[]

Model_Xtest=[Xp,Xcode,Xs,Xt,Xf]
for i,my_model in enumerate(models):
  model=keras.models.load_model('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/ablationsBERT_finall_{0}.keras'.format(models_names[i]))
  if(i==1):
    Model_Xtest=[Xp,Xs,Xt,Xf]
  if(i==2):
    Model_Xtest=[Xcode,Xs,Xt,Xf]
  if(i==3):
    Model_Xtest=[Xs,Xt,Xf]
  if(i==4):
    Model_Xtest=[Xp,Xcode]
  if(i==5):
    Model_Xtest=[Xcode]


  evaluate=model.evaluate(Model_Xtest,y)

  t1=time()
  y_pred=model.predict(Model_Xtest)
  t2=time()
  mcc=calculate_mcc_multiclass(y, y_pred)
  auc_pr=calculate_auc_pr_multiclass(y, y_pred)

  times_predict.append(t2-t1)
  accuracies_predict.append(evaluate[1])
  loss_predict.append(evaluate[0])
  mccs_predict.append(mcc)
  aucpr_predict.append(auc_pr)


  accuracy=evaluate[1]
  loss=evaluate[0]

  print("accuracy",accuracy)
  print("loss",loss)
  print("mcc",mcc)
  print("auc_pr",auc_pr)
  print("time predict",t2-t1)

2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 698ms/step - accuracy: 0.8735 - loss: 0.4953
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 869ms/step
accuracy 0.8571428656578064
loss 0.5434925556182861
mcc 0.6982378967958861
auc_pr 0.7095875013762771
time predict 1.8625869750976562


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 796ms/step - accuracy: 0.8735 - loss: 0.5525
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step
accuracy 0.8571428656578064
loss 0.6197311282157898
mcc 0.6996087759972994
auc_pr 0.5873588218752598
time predict 2.165205478668213


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 498ms/step - accuracy: 0.8440 - loss: 0.5491


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 440ms/step - accuracy: 0.8250 - loss: 0.5921
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 475ms/step
accuracy 0.800000011920929
loss 0.6732978224754333
mcc 0.5592680933870626
auc_pr 0.5062694831804224
time predict 0.9774253368377686


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 362ms/step - accuracy: 0.8735 - loss: 0.6101
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 329ms/step
accuracy 0.8571428656578064
loss 0.6779043674468994
mcc 0.7128514275273413
auc_pr 0.7584469824876771
time predict 0.6875603199005127


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [47]:
df_histories=pd.read_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsBERT/early_best_finall_predict.csv')
df_histories

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr
0,0,169.172873,1.000000,0.942238,2.029496,2.024967,0.000010,0.927746,0.254264,6.480963,0.903566,0.902499
1,1,117.749322,1.000000,0.944043,1.936619,2.035028,0.000001,0.919075,0.269747,5.492192,0.892148,0.895669
2,2,119.699390,1.000000,0.945848,2.111512,1.978953,0.000010,0.936416,0.257406,10.468127,0.915157,0.896689
3,3,109.264185,0.999548,0.942238,2.041958,2.024786,0.000100,0.920520,0.304918,1.285256,0.894048,0.890266
4,4,104.880635,0.998644,0.938628,1.994298,2.186902,0.000010,0.904624,0.446502,1.010328,0.873045,0.884939
5,5,72.745415,0.995029,0.933213,2.424156,2.173478,0.000010,0.903179,0.408474,1.028111,0.871218,0.885153


In [48]:
df_histories['accuracies_predictE']=accuracies_predict
df_histories['loss_predictE']=loss_predict
df_histories['times_predictE']=times_predict
df_histories['mccsE']=mccs_predict
df_histories['aucprE']=aucpr_predict
df_histories['model']=models_names

In [49]:
df_histories.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/bestFinallExternBERT.csv',index=False)

In [79]:
df_histories=pd.read_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/bestFinallExternBERT.csv')

In [80]:
df_histories

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model
0,0,169.172873,1.000000,0.942238,2.029496,2.024967,0.000010,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall
1,1,117.749322,1.000000,0.944043,1.936619,2.035028,0.000001,0.919075,0.269747,5.492192,0.892148,0.895669,0.857143,0.619731,2.165205,0.699609,0.587359,problem_gries
2,2,119.699390,1.000000,0.945848,2.111512,1.978953,0.000010,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries
3,3,109.264185,0.999548,0.942238,2.041958,2.024786,0.000100,0.920520,0.304918,1.285256,0.894048,0.890266,0.828571,0.517899,1.724972,0.631066,0.709759,gries
4,4,104.880635,0.998644,0.938628,1.994298,2.186902,0.000010,0.904624,0.446502,1.010328,0.873045,0.884939,0.800000,0.673298,0.977425,0.559268,0.506269,problem_code
5,5,72.745415,0.995029,0.933213,2.424156,2.173478,0.000010,0.903179,0.408474,1.028111,0.871218,0.885153,0.857143,0.677904,0.687560,0.712851,0.758447,code


In [81]:
df_histories[(df_histories['aucpr']>0.89)]

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model
0,0,169.172873,1.000000,0.942238,2.029496,2.024967,0.000010,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall
1,1,117.749322,1.000000,0.944043,1.936619,2.035028,0.000001,0.919075,0.269747,5.492192,0.892148,0.895669,0.857143,0.619731,2.165205,0.699609,0.587359,problem_gries
2,2,119.699390,1.000000,0.945848,2.111512,1.978953,0.000010,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries
3,3,109.264185,0.999548,0.942238,2.041958,2.024786,0.000100,0.920520,0.304918,1.285256,0.894048,0.890266,0.828571,0.517899,1.724972,0.631066,0.709759,gries


In [82]:
df_histories[(df_histories['aucpr']>0.89)&(df_histories['max_accuracies']-df_histories['max_val_accuracies']<0.06)]

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model
0,0,169.172873,1.000000,0.942238,2.029496,2.024967,0.000010,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall
1,1,117.749322,1.000000,0.944043,1.936619,2.035028,0.000001,0.919075,0.269747,5.492192,0.892148,0.895669,0.857143,0.619731,2.165205,0.699609,0.587359,problem_gries
2,2,119.699390,1.000000,0.945848,2.111512,1.978953,0.000010,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries
3,3,109.264185,0.999548,0.942238,2.041958,2.024786,0.000100,0.920520,0.304918,1.285256,0.894048,0.890266,0.828571,0.517899,1.724972,0.631066,0.709759,gries


# GRAPH CODE BEART

In [54]:
class Encoder:
  def __init__(self):
    self.is_loadtokenizers=False



  def tokenize_and_generate_embeddings_descriptions(self,description):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input description and move tokens to GPU
    tokens = self.beart_tokenizer.encode_plus(description, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.beart_model(**tokens)
    # Extract embeddings for all tokens
    desc_embeddings = outputs.last_hidden_state.cpu().numpy()
    return desc_embeddings

  def tokenize_and_generate_embeddings_codes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.codebeart_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.codebeart_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()

    return code_embeddings


  def tokenize_and_generate_embeddings_graphcodes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.graphcodebert_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.graphcodebert_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()
    return code_embeddings

  def load_beart_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    model_name = "bert-base-uncased"
    self.beart_model = BertModel.from_pretrained(model_name)
    self.beart_tokenizer = BertTokenizer.from_pretrained(model_name)
    self.beart_model.to(device)

  def load_graphcodebert_tokenizer(self):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    self.graphcodebert_tokenizer = AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")
    self.graphcodebert_model = AutoModel.from_pretrained("microsoft/graphcodebert-base")
    self.graphcodebert_model.to(device)


  def load_codebert_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    self.codebeart_tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
    self.codebeart_model = RobertaModel.from_pretrained("microsoft/codebert-base")
    self.codebeart_model.to(device)

  def start_tokenizer(self):
    self.load_beart_tokenizer()
    #self.load_codebert_tokenizer()
    self.load_graphcodebert_tokenizer()
    self.is_loadtokenizers=True




In [58]:
encoder=Encoder()
encoder.start_tokenizer()

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [61]:
%%time
problem=train_full['Problem Description'].apply(encoder.tokenize_and_generate_embeddings_descriptions).to_numpy()
code=train_full['Python Code'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
startstate = train_full['Estado incial'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
finalstate = train_full['Estado final'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
transstate = train_full['Transformación de estado'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()

CPU times: user 2.07 s, sys: 1.49 ms, total: 2.07 s
Wall time: 3.12 s


In [62]:
problem.shape,code.shape,startstate.shape,finalstate.shape,transstate.shape

((35,), (35,), (35,), (35,), (35,))

In [63]:
print(startstate.shape)
print(startstate[0].shape)


(35,)
(1, 12, 768)


In [64]:
Xp=np.array([sentence[0].mean(axis=0) for sentence in problem])
Xcode=np.array([sentence[0].mean(axis=0) for sentence in code])
Xs=np.array([sentence[0].mean(axis=0) for sentence in startstate])
Xf=np.array([sentence[0].mean(axis=0) for sentence in finalstate])
Xt=np.array([sentence[0].mean(axis=0) for sentence in transstate])
Xp.shape,Xs.shape,Xf.shape,Xt.shape

((35, 768), (35, 768), (35, 768), (35, 768))

In [65]:
y=train_full['Error Label']
y=y.to_numpy()
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

In [66]:
models=[]
models_names=["problemall","problem_gries","code_gries","gries","problem_code","code"]

problem_input=Input((768,))
code_input=Input((768,))
start_input=Input((768,))
trass_input=Input((768,))
final_input=Input((768,))

print("problem all")
models.append(neural_network_all(problem_input,code_input,start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3))
print("problem gries")
models.append(neural_network(problem_input,start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3))
print("problem code gries")
models.append(neural_network_code_gries(code_input,start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3))
print("gries")
models.append(neural_network_wproblem(start_input,trass_input,final_input,base=8,pow_initial=1, num_max_blocks=3))
print("problem code")
models.append(neural_network_problem_code(problem_input,code_input,base=8,pow_initial=1, num_max_blocks=3))
print("code")
models.append(neural_network_code(code_input,base=8,pow_initial=1, num_max_blocks=3))


problem all
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
problem gries
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
problem code gries
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
BottleNeck 64
Decoder 1
Block  0 512
base 8 pow 1 num_max_blocks 3
Encoder 1
Block  0 512
Bo

In [67]:
from tensorflow import keras
from time import time

accuracies_predict=[]
loss_predict=[]
times_predict=[]
mccs_predict=[]
aucpr_predict=[]

Model_Xtest=[Xp,Xcode,Xs,Xt,Xf]
for i,my_model in enumerate(models):
  model=keras.models.load_model('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/ablationsGRAPH_finall_{0}.keras'.format(models_names[i]))
  if(i==1):
    Model_Xtest=[Xp,Xs,Xt,Xf]
  if(i==2):
    Model_Xtest=[Xcode,Xs,Xt,Xf]
  if(i==3):
    Model_Xtest=[Xs,Xt,Xf]
  if(i==4):
    Model_Xtest=[Xp,Xcode]
  if(i==5):
    Model_Xtest=[Xcode]


  evaluate=model.evaluate(Model_Xtest,y)

  t1=time()
  y_pred=model.predict(Model_Xtest)
  t2=time()
  mcc=calculate_mcc_multiclass(y, y_pred)
  auc_pr=calculate_auc_pr_multiclass(y, y_pred)

  times_predict.append(t2-t1)
  accuracies_predict.append(evaluate[1])
  loss_predict.append(evaluate[0])
  mccs_predict.append(mcc)
  aucpr_predict.append(auc_pr)


  accuracy=evaluate[1]
  loss=evaluate[0]

  print("accuracy",accuracy)
  print("loss",loss)
  print("mcc",mcc)
  print("auc_pr",auc_pr)
  print("time predict",t2-t1)

2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 566ms/step - accuracy: 0.9030 - loss: 0.4376
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


accuracy 0.8857142925262451
loss 0.48994123935699463
mcc 0.7652281034846948
auc_pr 0.7981488675997754
time predict 2.6011464595794678
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 760ms/step - accuracy: 0.8440 - loss: 0.5803
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 763ms/step
accuracy 0.8285714387893677
loss 0.6432141065597534
mcc 0.6312719195444273
auc_pr 0.6692343840032555
time predict 1.86826753616333


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 503ms/step - accuracy: 0.9116 - loss: 0.2946
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 730ms/step
accuracy 0.9142857193946838
loss 0.2920931279659271
mcc 0.8259685381260148
auc_pr 0.9120644605837558
time predict 1.4922468662261963


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 385ms/step - accuracy: 0.9220 - loss: 0.2414
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 481ms/step
accuracy 0.9142857193946838
loss 0.25090399384498596
mcc 0.8253826429032017
auc_pr 0.9256944444444444
time predict 0.9795148372650146


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 308ms/step - accuracy: 0.9705 - loss: 0.1672
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 344ms/step
accuracy 0.9714285731315613
loss 0.1624256670475006
mcc 0.9434843870463827
auc_pr 0.9327039930555556
time predict 0.7016940116882324


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [68]:
df_historiesG=pd.read_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsGraph/ablation_best_finall_predict.csv')
df_historiesG

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735
1,1,106.195788,0.998644,0.940433,1.896729,1.993182,1.000000e-06,0.930636,0.297548,1.612740,0.907384,0.888445
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228
4,4,80.881043,0.985088,0.907942,2.009059,2.213736,1.000000e-07,0.903179,0.365639,1.042111,0.870503,0.886797
5,5,70.083852,0.992770,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874


In [69]:
df_historiesG['accuracies_predictE']=accuracies_predict
df_historiesG['loss_predictE']=loss_predict
df_historiesG['times_predictE']=times_predict
df_historiesG['mccsE']=mccs_predict
df_historiesG['aucprE']=aucpr_predict
df_historiesG['model']=models_names

In [72]:
df_historiesG.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/bestFinallExternGRAPH.csv',index=False)

In [73]:
df_historiesG=pd.read_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/bestFinallExternGRAPH.csv')

In [74]:
df_historiesG

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall
1,1,106.195788,0.998644,0.940433,1.896729,1.993182,1.000000e-06,0.930636,0.297548,1.612740,0.907384,0.888445,0.828571,0.643214,1.868268,0.631272,0.669234,problem_gries
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries
4,4,80.881043,0.985088,0.907942,2.009059,2.213736,1.000000e-07,0.903179,0.365639,1.042111,0.870503,0.886797,0.914286,0.250904,0.979515,0.825383,0.925694,problem_code
5,5,70.083852,0.992770,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code


In [104]:
df_historiesG[(df_historiesG['aucpr']>0.89)]

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries
5,5,70.083852,0.992770,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code


In [105]:
df_historiesG[(df_historiesG['aucpr']>0.89)&(df_historiesG['max_accuracies']-df_historiesG['max_val_accuracies']<0.06)]

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries
5,5,70.083852,0.992770,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code


# Analyze

In [96]:
import pandas as pd

In [97]:
df_bert=pd.read_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/bestFinallExternBERT.csv')
df_bert

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model
0,0,169.172873,1.000000,0.942238,2.029496,2.024967,0.000010,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall
1,1,117.749322,1.000000,0.944043,1.936619,2.035028,0.000001,0.919075,0.269747,5.492192,0.892148,0.895669,0.857143,0.619731,2.165205,0.699609,0.587359,problem_gries
2,2,119.699390,1.000000,0.945848,2.111512,1.978953,0.000010,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries
3,3,109.264185,0.999548,0.942238,2.041958,2.024786,0.000100,0.920520,0.304918,1.285256,0.894048,0.890266,0.828571,0.517899,1.724972,0.631066,0.709759,gries
4,4,104.880635,0.998644,0.938628,1.994298,2.186902,0.000010,0.904624,0.446502,1.010328,0.873045,0.884939,0.800000,0.673298,0.977425,0.559268,0.506269,problem_code
5,5,72.745415,0.995029,0.933213,2.424156,2.173478,0.000010,0.903179,0.408474,1.028111,0.871218,0.885153,0.857143,0.677904,0.687560,0.712851,0.758447,code


In [101]:
df_bert["enconder"]="bert"
df_bert

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
0,0,169.172873,1.000000,0.942238,2.029496,2.024967,0.000010,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall,bert
1,1,117.749322,1.000000,0.944043,1.936619,2.035028,0.000001,0.919075,0.269747,5.492192,0.892148,0.895669,0.857143,0.619731,2.165205,0.699609,0.587359,problem_gries,bert
2,2,119.699390,1.000000,0.945848,2.111512,1.978953,0.000010,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries,bert
3,3,109.264185,0.999548,0.942238,2.041958,2.024786,0.000100,0.920520,0.304918,1.285256,0.894048,0.890266,0.828571,0.517899,1.724972,0.631066,0.709759,gries,bert
4,4,104.880635,0.998644,0.938628,1.994298,2.186902,0.000010,0.904624,0.446502,1.010328,0.873045,0.884939,0.800000,0.673298,0.977425,0.559268,0.506269,problem_code,bert
5,5,72.745415,0.995029,0.933213,2.424156,2.173478,0.000010,0.903179,0.408474,1.028111,0.871218,0.885153,0.857143,0.677904,0.687560,0.712851,0.758447,code,bert


In [85]:
df_graph=pd.read_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/bestFinallExternGRAPH.csv')
df_graph


,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall
1,1,106.195788,0.998644,0.940433,1.896729,1.993182,1.000000e-06,0.930636,0.297548,1.612740,0.907384,0.888445,0.828571,0.643214,1.868268,0.631272,0.669234,problem_gries
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries
4,4,80.881043,0.985088,0.907942,2.009059,2.213736,1.000000e-07,0.903179,0.365639,1.042111,0.870503,0.886797,0.914286,0.250904,0.979515,0.825383,0.925694,problem_code
5,5,70.083852,0.992770,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code


In [102]:
df_graph["enconder"]="graph"
df_graph

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
1,1,106.195788,0.998644,0.940433,1.896729,1.993182,1.000000e-06,0.930636,0.297548,1.612740,0.907384,0.888445,0.828571,0.643214,1.868268,0.631272,0.669234,problem_gries,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries,graph
4,4,80.881043,0.985088,0.907942,2.009059,2.213736,1.000000e-07,0.903179,0.365639,1.042111,0.870503,0.886797,0.914286,0.250904,0.979515,0.825383,0.925694,problem_code,graph
5,5,70.083852,0.992770,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code,graph


In [103]:
# merge df_bert an df_graph
df_all=pd.concat([df_bert,df_graph])
df_all

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
0,0,169.172873,1.000000,0.942238,2.029496,2.024967,1.000000e-05,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall,bert
1,1,117.749322,1.000000,0.944043,1.936619,2.035028,1.000000e-06,0.919075,0.269747,5.492192,0.892148,0.895669,0.857143,0.619731,2.165205,0.699609,0.587359,problem_gries,bert
2,2,119.699390,1.000000,0.945848,2.111512,1.978953,1.000000e-05,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries,bert
3,3,109.264185,0.999548,0.942238,2.041958,2.024786,1.000000e-04,0.920520,0.304918,1.285256,0.894048,0.890266,0.828571,0.517899,1.724972,0.631066,0.709759,gries,bert
4,4,104.880635,0.998644,0.938628,1.994298,2.186902,1.000000e-05,0.904624,0.446502,1.010328,0.873045,0.884939,0.800000,0.673298,0.977425,0.559268,0.506269,problem_code,bert
5,5,72.745415,0.995029,0.933213,2.424156,2.173478,1.000000e-05,0.903179,0.408474,1.028111,0.871218,0.885153,0.857143,0.677904,0.687560,0.712851,0.758447,code,bert
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
1,1,106.195788,0.998644,0.940433,1.896729,1.993182,1.000000e-06,0.930636,0.297548,1.612740,0.907384,0.888445,0.828571,0.643214,1.868268,0.631272,0.669234,problem_gries,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries,graph


## Time

In [108]:
df_all[df_all['times']<df_all['times'].mean()]

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
3,3,109.264185,0.999548,0.942238,2.041958,2.024786,1.000000e-04,0.920520,0.304918,1.285256,0.894048,0.890266,0.828571,0.517899,1.724972,0.631066,0.709759,gries,bert
4,4,104.880635,0.998644,0.938628,1.994298,2.186902,1.000000e-05,0.904624,0.446502,1.010328,0.873045,0.884939,0.800000,0.673298,0.977425,0.559268,0.506269,problem_code,bert
5,5,72.745415,0.995029,0.933213,2.424156,2.173478,1.000000e-05,0.903179,0.408474,1.028111,0.871218,0.885153,0.857143,0.677904,0.687560,0.712851,0.758447,code,bert
1,1,106.195788,0.998644,0.940433,1.896729,1.993182,1.000000e-06,0.930636,0.297548,1.612740,0.907384,0.888445,0.828571,0.643214,1.868268,0.631272,0.669234,problem_gries,graph
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries,graph
4,4,80.881043,0.985088,0.907942,2.009059,2.213736,1.000000e-07,0.903179,0.365639,1.042111,0.870503,0.886797,0.914286,0.250904,0.979515,0.825383,0.925694,problem_code,graph
5,5,70.083852,0.992770,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code,graph


In [109]:

df_all[df_all['times_predict']<df_all['times_predict'].mean()]

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
3,3,109.264185,0.999548,0.942238,2.041958,2.024786,1.000000e-04,0.920520,0.304918,1.285256,0.894048,0.890266,0.828571,0.517899,1.724972,0.631066,0.709759,gries,bert
4,4,104.880635,0.998644,0.938628,1.994298,2.186902,1.000000e-05,0.904624,0.446502,1.010328,0.873045,0.884939,0.800000,0.673298,0.977425,0.559268,0.506269,problem_code,bert
5,5,72.745415,0.995029,0.933213,2.424156,2.173478,1.000000e-05,0.903179,0.408474,1.028111,0.871218,0.885153,0.857143,0.677904,0.687560,0.712851,0.758447,code,bert
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
1,1,106.195788,0.998644,0.940433,1.896729,1.993182,1.000000e-06,0.930636,0.297548,1.612740,0.907384,0.888445,0.828571,0.643214,1.868268,0.631272,0.669234,problem_gries,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries,graph
4,4,80.881043,0.985088,0.907942,2.009059,2.213736,1.000000e-07,0.903179,0.365639,1.042111,0.870503,0.886797,0.914286,0.250904,0.979515,0.825383,0.925694,problem_code,graph
5,5,70.083852,0.992770,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code,graph


In [107]:
df_all[df_all['loss_predict']<df_all['loss_predict'].mean()]

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
0,0,169.172873,1.000000,0.942238,2.029496,2.024967,1.000000e-05,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall,bert
1,1,117.749322,1.000000,0.944043,1.936619,2.035028,1.000000e-06,0.919075,0.269747,5.492192,0.892148,0.895669,0.857143,0.619731,2.165205,0.699609,0.587359,problem_gries,bert
2,2,119.699390,1.000000,0.945848,2.111512,1.978953,1.000000e-05,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries,bert
3,3,109.264185,0.999548,0.942238,2.041958,2.024786,1.000000e-04,0.920520,0.304918,1.285256,0.894048,0.890266,0.828571,0.517899,1.724972,0.631066,0.709759,gries,bert
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
1,1,106.195788,0.998644,0.940433,1.896729,1.993182,1.000000e-06,0.930636,0.297548,1.612740,0.907384,0.888445,0.828571,0.643214,1.868268,0.631272,0.669234,problem_gries,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries,graph


# Metrics Original Dataset

In [113]:
df_all[df_all['loss_predict']<df_all['loss_predict'].mean()]

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
0,0,169.172873,1.000000,0.942238,2.029496,2.024967,1.000000e-05,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall,bert
1,1,117.749322,1.000000,0.944043,1.936619,2.035028,1.000000e-06,0.919075,0.269747,5.492192,0.892148,0.895669,0.857143,0.619731,2.165205,0.699609,0.587359,problem_gries,bert
2,2,119.699390,1.000000,0.945848,2.111512,1.978953,1.000000e-05,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries,bert
3,3,109.264185,0.999548,0.942238,2.041958,2.024786,1.000000e-04,0.920520,0.304918,1.285256,0.894048,0.890266,0.828571,0.517899,1.724972,0.631066,0.709759,gries,bert
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
1,1,106.195788,0.998644,0.940433,1.896729,1.993182,1.000000e-06,0.930636,0.297548,1.612740,0.907384,0.888445,0.828571,0.643214,1.868268,0.631272,0.669234,problem_gries,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries,graph


In [115]:
df_all[df_all['accuracies_predict']>df_all['accuracies_predict'].mean()]

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
0,0,169.172873,1.000000,0.942238,2.029496,2.024967,1.000000e-05,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall,bert
2,2,119.699390,1.000000,0.945848,2.111512,1.978953,1.000000e-05,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries,bert
3,3,109.264185,0.999548,0.942238,2.041958,2.024786,1.000000e-04,0.920520,0.304918,1.285256,0.894048,0.890266,0.828571,0.517899,1.724972,0.631066,0.709759,gries,bert
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
1,1,106.195788,0.998644,0.940433,1.896729,1.993182,1.000000e-06,0.930636,0.297548,1.612740,0.907384,0.888445,0.828571,0.643214,1.868268,0.631272,0.669234,problem_gries,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries,graph


In [110]:
df_all[df_all['aucpr']>df_all['aucpr'].mean()]

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
0,0,169.172873,1.00000,0.942238,2.029496,2.024967,1.000000e-05,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall,bert
1,1,117.749322,1.00000,0.944043,1.936619,2.035028,1.000000e-06,0.919075,0.269747,5.492192,0.892148,0.895669,0.857143,0.619731,2.165205,0.699609,0.587359,problem_gries,bert
2,2,119.699390,1.00000,0.945848,2.111512,1.978953,1.000000e-05,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries,bert
0,0,144.717324,1.00000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
2,2,117.460625,1.00000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
5,5,70.083852,0.99277,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code,graph


In [111]:
df_all[df_all['mccs']>df_all['mccs'].mean()]

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
0,0,169.172873,1.000000,0.942238,2.029496,2.024967,1.000000e-05,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall,bert
2,2,119.699390,1.000000,0.945848,2.111512,1.978953,1.000000e-05,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries,bert
3,3,109.264185,0.999548,0.942238,2.041958,2.024786,1.000000e-04,0.920520,0.304918,1.285256,0.894048,0.890266,0.828571,0.517899,1.724972,0.631066,0.709759,gries,bert
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
1,1,106.195788,0.998644,0.940433,1.896729,1.993182,1.000000e-06,0.930636,0.297548,1.612740,0.907384,0.888445,0.828571,0.643214,1.868268,0.631272,0.669234,problem_gries,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries,graph


In [112]:
df_all[df_all['accuracies_predict']>df_all['accuracies_predict'].mean()]

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
0,0,169.172873,1.000000,0.942238,2.029496,2.024967,1.000000e-05,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall,bert
2,2,119.699390,1.000000,0.945848,2.111512,1.978953,1.000000e-05,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries,bert
3,3,109.264185,0.999548,0.942238,2.041958,2.024786,1.000000e-04,0.920520,0.304918,1.285256,0.894048,0.890266,0.828571,0.517899,1.724972,0.631066,0.709759,gries,bert
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
1,1,106.195788,0.998644,0.940433,1.896729,1.993182,1.000000e-06,0.930636,0.297548,1.612740,0.907384,0.888445,0.828571,0.643214,1.868268,0.631272,0.669234,problem_gries,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries,graph


## Metrics Extern Data

In [116]:
df_all[df_all['loss_predictE']<df_all['loss_predictE'].mean()]

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries,graph
4,4,80.881043,0.985088,0.907942,2.009059,2.213736,1.000000e-07,0.903179,0.365639,1.042111,0.870503,0.886797,0.914286,0.250904,0.979515,0.825383,0.925694,problem_code,graph
5,5,70.083852,0.992770,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code,graph


In [117]:
df_all[df_all['accuracies_predictE']>df_all['accuracies_predictE'].mean()]

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
4,4,80.881043,0.985088,0.907942,2.009059,2.213736,1.000000e-07,0.903179,0.365639,1.042111,0.870503,0.886797,0.914286,0.250904,0.979515,0.825383,0.925694,problem_code,graph
5,5,70.083852,0.992770,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code,graph


In [118]:
df_all[df_all['aucprE']>df_all['aucprE'].mean()]

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
2,2,119.699390,1.000000,0.945848,2.111512,1.978953,1.000000e-05,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries,bert
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries,graph
4,4,80.881043,0.985088,0.907942,2.009059,2.213736,1.000000e-07,0.903179,0.365639,1.042111,0.870503,0.886797,0.914286,0.250904,0.979515,0.825383,0.925694,problem_code,graph
5,5,70.083852,0.992770,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code,graph


In [120]:
df_all[df_all['aucprE']>df_all['aucprE'].mean()]

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
2,2,119.699390,1.000000,0.945848,2.111512,1.978953,1.000000e-05,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries,bert
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries,graph
4,4,80.881043,0.985088,0.907942,2.009059,2.213736,1.000000e-07,0.903179,0.365639,1.042111,0.870503,0.886797,0.914286,0.250904,0.979515,0.825383,0.925694,problem_code,graph
5,5,70.083852,0.992770,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code,graph


# The best metrics Original Dataset

In [147]:
df_all[df_all['loss_predict']<0.275].sort_values(by='loss_predict',ascending=False)

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
1,1,117.749322,1.0,0.944043,1.936619,2.035028,1.000000e-06,0.919075,0.269747,5.492192,0.892148,0.895669,0.857143,0.619731,2.165205,0.699609,0.587359,problem_gries,bert
0,0,144.717324,1.0,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
2,2,119.699390,1.0,0.945848,2.111512,1.978953,1.000000e-05,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries,bert
0,0,169.172873,1.0,0.942238,2.029496,2.024967,1.000000e-05,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall,bert
2,2,117.460625,1.0,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph


In [145]:
df_all[df_all['accuracies_predict']>0.927].sort_values(by='accuracies_predict',ascending=True)

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
2,2,119.699390,1.000000,0.945848,2.111512,1.978953,1.000000e-05,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries,bert
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
1,1,106.195788,0.998644,0.940433,1.896729,1.993182,1.000000e-06,0.930636,0.297548,1.612740,0.907384,0.888445,0.828571,0.643214,1.868268,0.631272,0.669234,problem_gries,graph
0,0,169.172873,1.000000,0.942238,2.029496,2.024967,1.000000e-05,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall,bert


In [149]:
df_all[df_all['mccs']>0.9].sort_values(by='mccs',ascending=True)

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
0,0,169.172873,1.000000,0.942238,2.029496,2.024967,1.000000e-05,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall,bert
1,1,106.195788,0.998644,0.940433,1.896729,1.993182,1.000000e-06,0.930636,0.297548,1.612740,0.907384,0.888445,0.828571,0.643214,1.868268,0.631272,0.669234,problem_gries,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
2,2,119.699390,1.000000,0.945848,2.111512,1.978953,1.000000e-05,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries,bert


In [193]:
df_all[df_all['aucpr']>0.8958].sort_values(by='aucpr',ascending=True)

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
5,5,70.083852,0.99277,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code,graph
2,2,119.699390,1.00000,0.945848,2.111512,1.978953,1.000000e-05,0.936416,0.257406,10.468127,0.915157,0.896689,0.828571,0.623299,1.488304,0.670481,0.780616,code_gries,bert
0,0,144.717324,1.00000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
2,2,117.460625,1.00000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
0,0,169.172873,1.00000,0.942238,2.029496,2.024967,1.000000e-05,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall,bert


# The best metrics Extern Dataset

In [200]:
df_all[df_all['loss_predictE']<0.5].sort_values(by='loss_predictE',ascending=False)

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
4,4,80.881043,0.985088,0.907942,2.009059,2.213736,1.000000e-07,0.903179,0.365639,1.042111,0.870503,0.886797,0.914286,0.250904,0.979515,0.825383,0.925694,problem_code,graph
5,5,70.083852,0.992770,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code,graph


In [263]:
df_all[df_all['accuracies_predictE']>0.8571].sort_values(by='accuracies_predictE',ascending=True)

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
0,0,169.172873,1.000000,0.942238,2.029496,2.024967,1.000000e-05,0.927746,0.254264,6.480963,0.903566,0.902499,0.857143,0.543493,1.862587,0.698238,0.709588,problemall,bert
1,1,117.749322,1.000000,0.944043,1.936619,2.035028,1.000000e-06,0.919075,0.269747,5.492192,0.892148,0.895669,0.857143,0.619731,2.165205,0.699609,0.587359,problem_gries,bert
5,5,72.745415,0.995029,0.933213,2.424156,2.173478,1.000000e-05,0.903179,0.408474,1.028111,0.871218,0.885153,0.857143,0.677904,0.687560,0.712851,0.758447,code,bert
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
4,4,80.881043,0.985088,0.907942,2.009059,2.213736,1.000000e-07,0.903179,0.365639,1.042111,0.870503,0.886797,0.914286,0.250904,0.979515,0.825383,0.925694,problem_code,graph
5,5,70.083852,0.992770,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code,graph


In [265]:
df_all[df_all['mccsE']>0.7].sort_values(by='mccsE',ascending=True)

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
5,5,72.745415,0.995029,0.933213,2.424156,2.173478,1.000000e-05,0.903179,0.408474,1.028111,0.871218,0.885153,0.857143,0.677904,0.687560,0.712851,0.758447,code,bert
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
4,4,80.881043,0.985088,0.907942,2.009059,2.213736,1.000000e-07,0.903179,0.365639,1.042111,0.870503,0.886797,0.914286,0.250904,0.979515,0.825383,0.925694,problem_code,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
5,5,70.083852,0.992770,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code,graph


In [272]:
df_all[df_all['aucprE']>0.79].sort_values(by='aucprE',ascending=True)

,Unnamed: 0,times,max_accuracies,max_val_accuracies,max_val_losses,max_losses,learning_rates,accuracies_predict,loss_predict,times_predict,mccs,aucpr,accuracies_predictE,loss_predictE,times_predictE,mccsE,aucprE,model,enconder
0,0,144.717324,1.000000,0.947653,1.957912,2.019822,1.000000e-07,0.934971,0.258696,2.008643,0.913152,0.896735,0.885714,0.489941,2.601146,0.765228,0.798149,problemall,graph
3,3,98.425345,0.998644,0.940433,1.953331,1.988816,1.000000e-06,0.920520,0.278513,1.281909,0.893815,0.892228,0.828571,0.423898,1.596869,0.653597,0.889341,gries,graph
2,2,117.460625,1.000000,0.953069,1.942038,2.025564,1.000000e-06,0.932081,0.250260,1.562257,0.909367,0.901699,0.914286,0.292093,1.492247,0.825969,0.912064,code_gries,graph
4,4,80.881043,0.985088,0.907942,2.009059,2.213736,1.000000e-07,0.903179,0.365639,1.042111,0.870503,0.886797,0.914286,0.250904,0.979515,0.825383,0.925694,problem_code,graph
5,5,70.083852,0.992770,0.936823,2.048632,2.196782,1.000000e-06,0.907514,0.378531,0.732815,0.876441,0.895874,0.971429,0.162426,0.701694,0.943484,0.932704,code,graph
